# Лабораторная лабора №4 - Проведение исследований со случайным лесом


### Подготовка

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sklearn
import keras
import nltk
import pandas as pd
import numpy as np
import re
import torch

from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.preprocessing import LabelEncoder

In [75]:
df = pd.read_excel("/content/drive/MyDrive/ai_course/dataset.xlsx")
df.to_csv("dataset.csv", index = False)
df

,oid,text,category
0,749208109,СПОЧНО СООБЩЕСТВО ПРОДАЕТСЯ ЗА 1300Р ЗА ПОКУПК...,esport
1,749208109,Пусть это побудет здесь БорьбаВпрямомЭфире How...,esport
2,749208109,Раздача пиздюлей от Мунсунга. HowToFtokenoid Б...,esport
3,749208109,Не знаю как вам но мне стилистика нравится пус...,esport
4,749208109,Скриншоты из новой главы. Тэхунчика показали и...,esport
...,...,...,...
53193,910636962,8 битная буря снова накрыла пикселями автомоби...,autosport
53194,669736851,Ира Сидоркова объясняет как сказалась на ее ма...,autosport
53195,558919241,24 я ракетка мира хорват Марин Чилич обыграл и...,tennis
53196,776944963,Стал известен календарь мужской сборной России...,volleyball


In [4]:
df_reg = pd.read_csv("/content/drive/MyDrive/ai_course/train.csv")
df_reg

,cow_id,milk_yield_kg,feed_energy_eke,feed_crude_protein_g,sugar_protein_ratio,breed,pasture_type,sire_breed,milk_fat_pct,milk_protein_pct,milk_taste_label,age_group
0,488,6005,13.5,1842,0.940,РефлешнСоверинг,Равнинное,Айдиал,3.62,3.073,не вкусно,более_2_лет
1,422,5982,13.8,1722,0.890,РефлешнСоверинг,Холмистое,Айдиал,3.61,3.073,не вкусно,более_2_лет
2,105,5700,14.4,1934,0.885,Вис Бик Айдиал,Равнинное,Соверин,3.57,3.079,вкусно,более_2_лет
3,115,5412,12.1,1924,0.890,Вис Бик Айдиал,Равнинное,Соверин,3.57,3.071,не вкусно,менее_2_лет
4,350,6171,15.3,1966,0.940,РефлешнСоверинг,Равнинное,Соверин,3.73,3.076,вкусно,более_2_лет
...,...,...,...,...,...,...,...,...,...,...,...,...
502,72,5809,13.5,2093,0.895,РефлешнСоверинг,Равнинные,Айдиалл,3.61,3.079,не вкусно,более_2_лет
503,107,5417,13.2,1848,0.890,Вис Бик Айдиал,Холмистое,Соверин,3.57,3.076,вкусно,менее_2_лет
504,271,6597,14.2,2204,0.950,РефлешнСоверинг,Равнинное,Соверин,3.74,3.079,не вкусно,более_2_лет
505,436,6054,15.7,1859,0.940,РефлешнСоверинг,Холмистое,Соверин,3.71,3.080,вкусно,более_2_лет


## 2. Создание бейзлайна и оценка качества

### Бейзлайн классификация

In [15]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53198 entries, 0 to 53197
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   oid       53198 non-null  int64 
 1   text      53198 non-null  object
 2   category  53198 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.2+ MB


In [76]:
RANDOM_STATE = 42

TEXT_COL = "text"
LABEL_COL = "category"

In [77]:
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
df

,oid,text,category
0,749208109,СПОЧНО СООБЩЕСТВО ПРОДАЕТСЯ ЗА 1300Р ЗА ПОКУПК...,esport
1,749208109,Пусть это побудет здесь БорьбаВпрямомЭфире How...,esport
2,749208109,Раздача пиздюлей от Мунсунга. HowToFtokenoid Б...,esport
3,749208109,Не знаю как вам но мне стилистика нравится пус...,esport
4,749208109,Скриншоты из новой главы. Тэхунчика показали и...,esport
...,...,...,...
53193,910636962,8 битная буря снова накрыла пикселями автомоби...,autosport
53194,669736851,Ира Сидоркова объясняет как сказалась на ее ма...,autosport
53195,558919241,24 я ракетка мира хорват Марин Чилич обыграл и...,tennis
53196,776944963,Стал известен календарь мужской сборной России...,volleyball


In [78]:
le = LabelEncoder()
y = le.fit_transform(df[LABEL_COL].astype(str))
X = df[TEXT_COL].astype(str)

In [79]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

In [80]:
classification = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=80_000,
        sublinear_tf=True,
        norm="l2"
    )),
    ("rf", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

In [24]:
classification.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, max_features=80000, min_df=2,
                                 ngram_range=(1, 2), sublinear_tf=True)),
                ('rf', RandomForestClassifier(n_jobs=-1, random_state=42))])

In [25]:
y_pred = classification.predict(X_test)

In [26]:
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

   athletics       0.92      0.87      0.89       957
   autosport       0.90      0.77      0.83       630
  basketball       0.88      0.77      0.82       866
  boardgames       0.87      0.93      0.90       910
      esport       0.58      0.79      0.67       902
     extreme       0.67      0.59      0.63       634
    football       0.58      0.75      0.65       852
      hockey       0.92      0.45      0.61       370
martial_arts       0.78      0.71      0.74       852
   motosport       0.88      0.90      0.89       906
      tennis       0.94      0.92      0.93       917
  volleyball       0.92      0.78      0.85       937
winter_sport       0.77      0.83      0.80       907

    accuracy                           0.80     10640
   macro avg       0.82      0.77      0.79     10640
weighted avg       0.81      0.80      0.80     10640



#### Выводы по бейзлайну

Результат сильно лучше, чем на маленьких моделях, было около 0.61

Похожие классы все еще мало различимы

### Бейзлайн регрессия

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [6]:
RANDOM_STATE=42

In [7]:
train = pd.read_csv("/content/drive/MyDrive/ai_course/train.csv")
test = pd.read_csv("/content/drive/MyDrive/ai_course/test.csv")
train.head()

,cow_id,milk_yield_kg,feed_energy_eke,feed_crude_protein_g,sugar_protein_ratio,breed,pasture_type,sire_breed,milk_fat_pct,milk_protein_pct,milk_taste_label,age_group
0,488,6005,13.5,1842,0.940,РефлешнСоверинг,Равнинное,Айдиал,3.62,3.073,не вкусно,более_2_лет
1,422,5982,13.8,1722,0.890,РефлешнСоверинг,Холмистое,Айдиал,3.61,3.073,не вкусно,более_2_лет
2,105,5700,14.4,1934,0.885,Вис Бик Айдиал,Равнинное,Соверин,3.57,3.079,вкусно,более_2_лет
3,115,5412,12.1,1924,0.890,Вис Бик Айдиал,Равнинное,Соверин,3.57,3.071,не вкусно,менее_2_лет
4,350,6171,15.3,1966,0.940,РефлешнСоверинг,Равнинное,Соверин,3.73,3.076,вкусно,более_2_лет


In [8]:
TARGET_COL = "milk_yield_kg"
ID_COL_CANDIDATES = ["cow_id"]

In [9]:
feature_cols = [c for c in train.columns if c != TARGET_COL]

id_col = next((c for c in ID_COL_CANDIDATES if c in test.columns), None)
if id_col is None:
    test_id = pd.Series(test.index, name="row_id")
else:
    test_id = test[id_col].copy()

In [10]:
num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train[c])]
cat_cols = [c for c in feature_cols if c not in num_cols]

Минимальные препроцессинг данных для работы модели

In [11]:
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ],
    remainder="drop"
)

In [12]:
reg = Pipeline(steps=[
    ("prep", preprocess),
    ("rf", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

In [13]:
X = train[feature_cols]
y = train[TARGET_COL].astype(float)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

In [14]:
reg.fit(X_tr, y_tr)
y_hat = reg.predict(X_te)

mae  = mean_absolute_error(y_te, y_hat)
mse = mean_squared_error(y_te, y_hat)
r2 = r2_score(y_te, y_hat)
print(f"MAE : {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"R^2 : {r2:.4f}")

MAE : 367.3829
MSE: 6369953.4244
R^2 : -22.9124


#### Выводы по регрессии

Качество среднее: отрицательный R2 означает, что модель предсказывает хуже, чем простое среднее значение.

На маленький модельках качество бейзлайна явно хуже

## 3. Улучшение бейзлайна

### Классификация

Гипотезы для улучшения:

- Увеличение n_estimators повысит стабильность и качество

- Ограничение max_depth снизит переобучение

- Подбор min_samples_leaf улучшит обобщающую способность

В целом качество будет выше бейзлайна, но обучение станет медленнее

#### Загрузка данных

Подгружаю данные, которые доработывала и добавляла фичи из 1 лр

In [53]:
import numpy as np
from joblib import load

X_train_embeddings = np.load("/content/drive/MyDrive/ai_course/X_train_embeddings.npy")
X_test_embeddings  = np.load("/content/drive/MyDrive/ai_course/X_test_embeddings.npy")
y_train_classification = np.load("/content/drive/MyDrive/ai_course/y_train.npy")
y_test_classification = np.load("/content/drive/MyDrive/ai_course/y_test.npy")

In [28]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 27.8 MB/s eta 0:00:00


#### Обучение модели

In [29]:
RANDOM_STATE = 42

In [30]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score

In [54]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 5, 50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestClassifier(**params)
    model.fit(X_train_embeddings, y_train_classification)

    y_pred = model.predict(X_test_embeddings)
    return accuracy_score(y_test_classification, y_pred)

In [57]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best params:", study.best_params)
print("Best accuracy:", study.best_value)

[I 2025-12-14 13:17:26,721] A new study created in memory with name: no-name-dc9fc5bc-b188-458e-968a-7b71bc7787ff
[I 2025-12-14 13:18:49,136] Trial 0 finished with value: 0.7486897928624906 and parameters: {'n_estimators': 160, 'max_depth': 40, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7486897928624906.
[I 2025-12-14 13:20:57,071] Trial 1 finished with value: 0.7529323683553781 and parameters: {'n_estimators': 249, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.7529323683553781.
[I 2025-12-14 13:21:32,955] Trial 2 finished with value: 0.7154978787122536 and parameters: {'n_estimators': 281, 'max_depth': 9, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 1 with value: 0.7529323683553781.
[I 2025-12-14 13:24:17,493] Trial 3 finished with value: 0.7466932867481907 and parameters: {'n_estimators': 341, 'max_depth': 50, 'mi

Best params: {'n_estimators': 249, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Best accuracy: 0.7529323683553781


In [61]:
best_rf = RandomForestClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train_embeddings, y_train_classification)

y_pred = best_rf.predict(X_test_embeddings)

print("Final Acc:", accuracy_score(y_test_classification, y_pred))
print(classification_report(y_test_classification, y_pred))

Final Acc: 0.7529323683553781
              precision    recall  f1-score   support

           0       0.79      0.80      0.79       311
           1       0.80      0.67      0.73       302
           2       0.77      0.67      0.72       307
           3       0.88      0.87      0.88       305
           4       0.69      0.81      0.75       314
           5       0.56      0.75      0.64       306
           6       0.80      0.66      0.72       307
           7       0.58      0.75      0.66       301
           8       0.90      0.75      0.82       313
           9       0.81      0.68      0.74       303
          10       0.87      0.86      0.86       317
          11       0.80      0.72      0.76       316
          12       0.71      0.80      0.75       305

    accuracy                           0.75      4007
   macro avg       0.77      0.75      0.75      4007
weighted avg       0.77      0.75      0.76      4007



#### ИТОГ 3 пункта

Метрики на бейзлайне
```
    accuracy                           0.80     10640
   macro avg       0.82      0.77      0.79     10640
weighted avg       0.81      0.80      0.80     10640
```

качество стало хуже, скорее всего это из-за того, что эмбеддинги в данной задаче срабатывают хуже, чем tf-idf

### Регрессия

Попробуем:

1. Улучшенные данные увеличат метрику: чистые признаки -> меньше шума, модель будет меньше переобучаться, MAE сразу уменьшится

2. Подбор гиперпараметров Random Forest

Ожидание: Ошибка (MAE, MSE) снизится

#### Загрузка чистых данных

In [33]:
X_test = pd.read_csv("/content/drive/MyDrive/ai_course/X_test.csv")
X_train = pd.read_csv("/content/drive/MyDrive/ai_course/X_train.csv")
y_train = pd.read_csv("/content/drive/MyDrive/ai_course/y_train.csv").squeeze()
y_test  = pd.read_csv("/content/drive/MyDrive/ai_course/y_test.csv").squeeze()

#### Обучение

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 12.1 MB/s eta 0:00:00


In [34]:
RANDOM_STATE=42

In [35]:
import numpy as np
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone

In [36]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop",
)

In [39]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 5, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state": 42,
        "n_jobs": -1
    }

    model = Pipeline(steps=[
        ("prep", preprocess),
        ("rf", RandomForestRegressor(**params))
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return rmse

In [41]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best params:", study.best_params)
print("Best RMSE:", study.best_value)

[I 2025-12-14 13:08:50,149] A new study created in memory with name: no-name-0f821af5-b4bb-44dc-9d05-dd0c7392db58
[I 2025-12-14 13:08:50,980] Trial 0 finished with value: 165.20288487552952 and parameters: {'n_estimators': 468, 'max_depth': 28, 'min_samples_split': 3, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 165.20288487552952.
[I 2025-12-14 13:08:51,498] Trial 1 finished with value: 164.2105877962984 and parameters: {'n_estimators': 268, 'max_depth': 6, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 1 with value: 164.2105877962984.
[I 2025-12-14 13:08:52,333] Trial 2 finished with value: 163.29729767072695 and parameters: {'n_estimators': 440, 'max_depth': 33, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 2 with value: 163.29729767072695.
[I 2025-12-14 13:08:53,205] Trial 3 finished with value: 158.18678379489702 and parameters: {'n_estimators': 493, 'max_depth': 16, 'min_

Best params: {'n_estimators': 210, 'max_depth': 30, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
Best RMSE: 154.6846990587421


In [43]:
best_reg_rf = Pipeline(steps=[
    ("prep", preprocess),
    ("rf", RandomForestRegressor(
        **study.best_params,
        random_state=42,
        n_jobs=-1
    ))
])

best_reg_rf.fit(X_train, y_train)

y_pred = best_reg_rf.predict(X_test)

print("Final RMSE:", mean_squared_error(y_test, y_pred))
print("Final MAE:", mean_absolute_error(y_test, y_pred))
print("Final R2:", r2_score(y_test, y_pred))

Final RMSE: 23927.356122893605
Final MAE: 122.71942110177402
Final R2: 0.8992851720012845


#### ИТОГ РЕГРЕССИЯ


Напоминаю старые метрики

```
MAE : 367.3829
MSE: 6369953.4244
R^2 : -22.9124
```




В бейзлайне деревья практически не справлялись с задачей, сейчас - после очистки и подбора парамтеров получаем очень хорошие метрики. R2 уже полодительная и неплохая, mae почти лучшее, если сравниваться с ЛР1-ЛР3

## 4. Имплементация алгоритма машинного обучения

### Классификация

Гипотеза: буду использовать tf-idf, результат должен быть похож на бейзлайн

In [65]:
import numpy as np
from joblib import load

# X_train_embeddings = np.load("/content/drive/MyDrive/ai_course/X_train_embeddings.npy")
# X_test_embeddings  = np.load("/content/drive/MyDrive/ai_course/X_test_embeddings.npy")
# y_train = np.load("/content/drive/MyDrive/ai_course/y_train.npy")
# y_test = np.load("/content/drive/MyDrive/ai_course/y_test.npy")

In [82]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import numpy as np


class RandomForestTextClassifier:
    def __init__(
        self,
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42
    ):

        self.params = {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_split": min_samples_split,
            "min_samples_leaf": min_samples_leaf,
            "max_features": max_features,
            "random_state": random_state,
            "n_jobs": -1
        }

        # pipeline
        self.model = Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                max_features=80_000,
                sublinear_tf=True,
                norm="l2"
            )),
            ("rf", RandomForestClassifier(**self.params))
        ])

        self.is_fitted = False

    def fit(self, X, y):
        self.model.fit(X, y)
        self.is_fitted = True

    def predict(self, X):
        return self.model.predict(X)

    def evaluate(self, X, y):
        y_pred = self.predict(X)
        acc = accuracy_score(y, y_pred)
        report = classification_report(y, y_pred)
        return acc, report

    def get_feature_importance(self, top_k=20):
        rf = self.model.named_steps["rf"]
        importances = rf.feature_importances_

        return np.sort(importances)[-top_k:][::-1]

In [83]:
clf = RandomForestTextClassifier(
    n_estimators=249,
    max_depth=None,
    max_features="sqrt"
)

In [84]:
clf.fit(X_train, y_train)

acc, report = clf.evaluate(X_test, y_test)

print(report)

              precision    recall  f1-score   support

           0       0.92      0.86      0.89       957
           1       0.91      0.77      0.84       630
           2       0.90      0.78      0.83       866
           3       0.87      0.93      0.90       910
           4       0.59      0.81      0.68       902
           5       0.68      0.60      0.64       634
           6       0.59      0.77      0.67       852
           7       0.92      0.46      0.62       370
           8       0.78      0.72      0.75       852
           9       0.88      0.90      0.89       906
          10       0.94      0.92      0.93       917
          11       0.92      0.79      0.85       937
          12       0.78      0.83      0.80       907

    accuracy                           0.80     10640
   macro avg       0.82      0.78      0.79     10640
weighted avg       0.82      0.80      0.80     10640



#### ИТОГ 4 ПУНКТ


Собственная реализация показала результат действительно похожий на бейзлайн
```
    accuracy                           0.80     10640
   macro avg       0.82      0.77      0.79     10640
weighted avg       0.81      0.80      0.80     10640
```

### Регрессия

Гипотезы: наша модель должна давать близкие метрики к sklearn и сильно лучше бейзлайна

In [44]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.utils import resample
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

class SimpleRandomForestRegressor:
    def __init__(self, n_estimators=10, max_depth=None, min_samples_split=2,
                 min_samples_leaf=1, max_features=None, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.features_idx = []

    def fit(self, X, y):
        np.random.seed(self.random_state)
        self.trees = []
        self.features_idx = []
        n_features = X.shape[1]
        max_features = self.max_features or int(np.sqrt(n_features))

        for i in range(self.n_estimators):
            # Bootstrap sample
            X_sample, y_sample = resample(X, y, replace=True, random_state=self.random_state+i)

            # Случайный выбор признаков
            features = np.random.choice(n_features, max_features, replace=False)
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                random_state=self.random_state+i
            )
            tree.fit(X_sample[:, features], y_sample)

            self.trees.append(tree)
            self.features_idx.append(features)

    def predict(self, X):
        preds = np.zeros((X.shape[0], self.n_estimators))
        for i, tree in enumerate(self.trees):
            preds[:, i] = tree.predict(X[:, self.features_idx[i]])
        return np.mean(preds, axis=1)

In [45]:
X_train_enc = preprocess.fit_transform(X_train)
X_test_enc  = preprocess.transform(X_test)

X_train_arr = X_train_enc.toarray() if hasattr(X_train_enc, "toarray") else X_train_enc
X_test_arr  = X_test_enc.toarray() if hasattr(X_test_enc, "toarray") else X_test_enc

y_train_arr = np.asarray(y_train, dtype=float)
y_test_arr  = np.asarray(y_test, dtype=float)

In [48]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = SimpleRandomForestRegressor(
    n_estimators=210,
    max_depth=30,
    min_samples_leaf=2,
    max_features=None,
    random_state=42
)
model.fit(X_train_arr, y_train_arr)
y_pred_np = model.predict(X_test_arr)

In [49]:
mae_np = mean_absolute_error(y_test_arr, y_pred_np)
mse_np = mean_squared_error(y_test_arr, y_pred_np)
r2_np = r2_score(y_test_arr, y_pred_np)

print(f"MAE: {mae_np:.4f}")
print(f"MSE: {mse_np:.4f}")
print(f"R^2: {r2_np:.4f}")

MAE : 228.9462
MSE: 74922.0996
R^2 : 0.6846


#### ИТОГ РЕГРЕССИЯ

Результат, как и ожидалось, получился лучше бейзлайна в 2 раза, но хуже, чем с optuna